# DPR (2020)
---
[[paper]](https://arxiv.org/abs/2004.04906)<br>
DPR = Dense Passage Retrieval

__DPR__ — это метод Dense Retrieval, основанный на архитектуре Bi-encoder, который преобразует текстовые запросы и документы в плотные векторные представления для поиска по семантическому сходству в векторном пространстве.

__Постановка задачи__<br>
Решается задача Open-Domain Question Answering (ODQA). По заданному вопросу $q$ необходимо найти наиболее релевантный фрагмент текста (passage) $p$ из огромной коллекции документов (например, всей Википедии), который содержит ответ на этот вопрос.

__Мотивация__<br>
Классические методы Sparse Retrieval, такие как BM25 (2009) или TF-IDF, основаны на лексическом совпадении слов. Они плохо справляются с поиском, если вопрос и документ используют разные синонимы или если запрос требует понимания смысла, а не просто поиска ключевых слов. Задача состояла в том, чтобы научить модель находить релевантные тексты на основе их семантического содержания, используя при этом преимущество предварительно обученных языковых моделей.

__Существующие подходы__<br>
До появления DPR основными методами были:
- BM25: использует статистику совпадения термов. Минус — полное отсутствие понимания семантики.
- ORQA (2019): первая серьезная попытка Dense Retrieval для QA. Использовала Inverse Cloze Task (ICT) для предобучения. Минус — сложная и вычислительно дорогая процедура обучения, зависимость от предобучения на скрытых задачах.
- REALM (2020): объединяет ретривер и ридер (модель для извлечения ответа) в одну систему. Минус — требует огромных ресурсов для совместного обучения и периодического перестроения индекса всей коллекции во время обучения.

__Идея__<br>
Авторы предложили доказать, что простое обучение Bi-encoder архитектуры на относительно небольшом наборе пар "вопрос-ответ" с использованием правильной стратегии выбора негативных примеров (Negative Sampling) может превзойти и BM25, и сложные системы вроде ORQA. Основная новизна заключалась в использовании Hard Negatives — документов, которые лексически похожи на запрос (высокий скор по BM25), но не содержат правильного ответа.

__Архитектура__<br>
DPR использует два независимых трансформера (обычно BERT-base, 2018):
1.  Question Encoder ($E_Q$): принимает текст вопроса и выдает векторное представление $v_q$ (обычно это эмбеддинг [CLS] токена).
2.  Passage Encoder ($E_P$): принимает текст документа и выдает векторное представление $v_p$.
3.  Similarity Function: сходство между вопросом и документом определяется как скалярное произведение (Inner Product) их векторов: $sim(q, p) = E_Q(q)^T E_P(p)$.
4.  Размерность векторов: 768 (для BERT-base).

__Алгоритм обучения__<br>
Цель обучения — максимизировать сходство между вопросом и позитивным (релевантным) документом и минимизировать сходство с негативными документами.
1.  Dataset: Набор данных $\mathcal{D} = \{ (q_i, p_i^+, p_{i,1}^-, \dots, p_{i,n}^-) \}$, где $p^+$ — релевантный документ, а $p^-$ — нерелевантные.
2.  Loss Function: Negative Log-Likelihood (NLL) позитивного документа относительно всех документов в батче.
3.  In-batch Negatives: для эффективного вычисления используются документы других вопросов в рамках одного батча. Если в батче $B$ вопросов, то для каждого вопроса есть 1 позитивный и $(B-1)$ негативных примера "бесплатно".
4.  Hard Negatives: К батчу добавляется один "сложный" негативный пример, найденный через BM25 (высокий BM25 скор, но нет ответа). Это критически важный этап, который заставляет модель различать тонкие семантические нюансы, а не просто реагировать на совпадение слов.

__Алгоритм инференса__<br>
1.  Offline (индексация): Все документы коллекции пропускаются через обученный $E_P$. Полученные векторы сохраняются в индекс для быстрого поиска, например, FAISS (библиотека для Efficient Similarity Search).
2.  Online (поиск): При поступлении запроса он пропускается через $E_Q$ для получения вектора $v_q$.
3.  Retrieval: В FAISS выполняется поиск Top-K ближайших векторов к $v_q$ по метрике Inner Product.
4.  Результат: Возвращаются тексты, соответствующие найденным векторам.

__Результаты__<br>
Эксперименты проводились на датасетах Natural Questions (NQ) и TriviaQA:
- На датасете NQ метрика Top-20 Retrieval Accuracy (вероятность того, что правильный ответ есть в топ-20 выданных документов) составила 79.4%, что на 20пп выше, чем у BM25 (59.1%).
- DPR превзошел модель ORQA на 11пп по точности Top-20, используя при этом в разы меньше вычислительных ресурсов для обучения.
- Было доказано, что комбинация DPR + BM25 (Hybrid Search) дает еще лучший результат, но даже "чистый" DPR значительно опережает классические методы в задачах поиска ответов на вопросы.

## 📝 Критический анализ

```markdown
# DPR (2020)
---
[[paper]](https://arxiv.org/abs/2004.04906)<br>
DPR = Dense Passage Retrieval

__DPR__ — метод Dense Retrieval, использующий Bi-encoder для преобразования текстовых запросов и документов в векторные представления, что позволяет искать по семантическому сходству.

__Постановка задачи__<br>
Решается задача Open-Domain Question Answering (ODQA): по вопросу $q$ найти релевантный фрагмент текста $p$ из большой коллекции документов.

__Мотивация__<br>
Классические методы, такие как BM25, основаны на лексическом совпадении и не справляются с семантическим поиском. Задача — научить модель находить релевантные тексты на основе семантики, используя преимущества языковых моделей.

__Существующие подходы__<br>
- BM25: основан на совпадении термов, не учитывает семантику.
- ORQA (2019): использует Inverse Cloze Task для предобучения, но сложен и ресурсоемок.
- REALM (2020): объединяет ретривер и ридер, требует больших ресурсов.

__Идея__<br>
Обучение Bi-encoder на небольшом наборе пар "вопрос-ответ" с использованием Hard Negatives — документов, лексически похожих на запрос, но не содержащих ответа.

__Архитектура__<br>
DPR использует два трансформера (обычно BERT-base):
1.  Question Encoder ($E_Q$): преобразует вопрос в вектор $v_q$.
2.  Passage Encoder ($E_P$): преобразует документ в вектор $v_p$.
3.  Similarity Function: скалярное произведение векторов $sim(q, p) = E_Q(q)^T E_P(p)$.

<img src="img/img.png" width=500>

__Алгоритм обучения__<br>
1.  Dataset: $\mathcal{D} = \{ (q_i, p_i^+, p_{i,1}^-, \dots, p_{i,n}^-) \}$.
2.  Loss Function: Negative Log-Likelihood (NLL) позитивного документа.
3.  In-batch Negatives: документы других вопросов в батче.
4.  Hard Negatives: добавление "сложного" негативного примера через BM25.

__Алгоритм инференса__<br>
1.  Offline: индексация документов через $E_P$.
2.  Online: запрос пропускается через $E_Q$.
3.  Retrieval: поиск Top-K векторов в FAISS.
4.  Результат: возвращаются тексты, соответствующие векторам.

__Результаты__<br>
- На Natural Questions Top-20 Retrieval Accuracy достигла 79.4%, что на 20пп выше BM25.
- DPR превзошел ORQA на 11пп по точности Top-20, используя меньше ресурсов.
- Комбинация DPR + BM25 улучшает результаты, но даже "чистый" DPR значительно опережает классические методы.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импортируем необходимые библиотеки
from transformers import DPRQuestionEncoder, DPRContextEncoder, DPRQuestionEncoderTokenizer, DPRContextEncoderTokenizer
from transformers import DPRQuestionEncoderTokenizerFast, DPRContextEncoderTokenizerFast
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Инициализируем токенайзеры и энкодеры для вопросов и документов
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

question_tokenizer = DPRQuestionEncoderTokenizerFast.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
context_tokenizer = DPRContextEncoderTokenizerFast.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

# Пример вопроса и документов
question = "What is the capital of France?"
passages = [
    "Paris is the capital of France.",
    "Berlin is the capital of Germany.",
    "Madrid is the capital of Spain."
]

# Кодируем вопрос и документы в векторные представления
question_inputs = question_tokenizer(question, return_tensors="pt")
question_embedding = question_encoder(**question_inputs).pooler_output

passage_embeddings = []
for passage in passages:
    passage_inputs = context_tokenizer(passage, return_tensors="pt")
    passage_embedding = context_encoder(**passage_inputs).pooler_output
    passage_embeddings.append(passage_embedding)

# Конвертируем список тензоров в один тензор
passage_embeddings = torch.cat(passage_embeddings, dim=0)

# Вычисляем сходство между вопросом и каждым из документов
similarities = cosine_similarity(question_embedding.detach().numpy(), passage_embeddings.detach().numpy())

# Находим индекс документа с максимальным сходством
most_similar_idx = np.argmax(similarities)

# Выводим наиболее релевантный документ
print(f"Most relevant passage: {passages[most_similar_idx]}")

# Выводим сходства для всех документов
print("Similarities:", similarities)
```

### Комментарии к коду:

1. **Импорт библиотек**: Используем библиотеку `transformers` от Hugging Face, которая предоставляет готовые реализации моделей DPR.

2. **Инициализация моделей и токенайзеров**: Загружаем предобученные модели и токенайзеры для вопросов и документов. Это соответствует архитектуре Bi-encoder, где используются два независимых энкодера.

3. **Пример данных**: Вопрос и несколько документов (passages), среди которых нужно найти наиболее релевантный.

4. **Кодирование**: Преобразуем текстовые данные в векторные представления с помощью токенайзеров и энкодеров. Векторное представление вопроса и каждого документа извлекается из `pooler_output`.

5. **Сходство**: Вычисляем косинусное сходство между вектором вопроса и векторами документов. Это иллюстрирует использование скалярного произведения для определения семантической близости.

6. **Поиск наиболее релевантного документа**: Находим индекс документа с максимальным сходством и выводим его. Это демонстрирует процесс инференса, где для заданного вопроса мы ищем наиболее подходящий документ.

Этот пример иллюстрирует основные концепции DPR, такие как использование Bi-encoder архитектуры и семантическое сопоставление через векторные представления.